# Clase 109 — Activaciones: ReLU, ELU, GELU, Swish, Mish

Familia de activaciones modernas: desde **ReLU** (`max(0,x)`) hasta **GELU** (default en Transformers) y **Swish/SiLU** (`x·σ(x)`, EfficientNet). Elegir según arquitectura y entender el **dying ReLU**.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `matplotlib`.

## 1. Las funciones evaluadas en un rango

Keras 3 expone todas: `relu`, `elu`, `gelu`, `silu` (= swish), `mish`, más `LeakyReLU` como capa.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

x = tf.linspace(-3.0, 3.0, 7)
print("ReLU :", keras.activations.relu(x).numpy().round(3))
print("ELU  :", keras.activations.elu(x).numpy().round(3))
print("GELU :", keras.activations.gelu(x).numpy().round(3))
print("SiLU :", keras.activations.silu(x).numpy().round(3))   # = Swish
print("Mish :", keras.activations.mish(x).numpy().round(3))
leaky = layers.LeakyReLU(negative_slope=0.1)                  # Keras 3: negative_slope
print("Leaky:", leaky(x).numpy().round(3))

## 2. Definiciones matemáticas a mano

`GELU = x·Φ(x)` (Φ = CDF gaussiana, vía `erf`); `Swish = x·σ(x)`; `Mish = x·tanh(softplus(x))`.

In [ ]:
def gelu_manual(z):
    return 0.5 * z * (1.0 + tf.math.erf(z / tf.sqrt(2.0)))    # GELU exacta
def swish_manual(z):
    return z * tf.sigmoid(z)
def mish_manual(z):
    return z * tf.tanh(tf.math.softplus(z))

z = tf.constant([-2.0, 0.0, 2.0])
print("GELU manual:", gelu_manual(z).numpy().round(4),
      "| Keras:", keras.activations.gelu(z).numpy().round(4))
print("Swish manual:", swish_manual(z).numpy().round(4))
print("Mish  manual:", mish_manual(z).numpy().round(4))

## 3. Comparación empírica entre MLPs

Mismo init He, mismo LR; solo cambia la activación de las capas ocultas.

In [ ]:
def mlp(activacion):
    m = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(256, activation=activacion, kernel_initializer="he_normal"),
        layers.Dense(128, activation=activacion, kernel_initializer="he_normal"),
        layers.Dense(64,  activation=activacion, kernel_initializer="he_normal"),
        layers.Dense(10,  activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

for act in ["relu", "elu", "gelu", "swish"]:
    print(f"{act:6s} params={mlp(act).count_params()}")
    # mlp(act).fit(X_tr, y_tr, epochs=15, validation_split=0.1)

## 4. Dying ReLU: contar neuronas muertas

Una neurona ReLU "muerta" tiene salida 0 para todo el batch → gradiente 0 → nunca se actualiza.

In [ ]:
inp = keras.Input(shape=(784,))
out = layers.Dense(256, activation="relu", kernel_initializer="he_normal")(inp)
sonda = keras.Model(inp, out)

batch = np.random.default_rng(0).normal(size=(1024, 784)).astype("float32")
act = sonda(batch).numpy()
muertas = float(np.mean(act.mean(axis=0) == 0.0))
print(f"neuronas con salida 0 en todo el batch (ReLU): {muertas:.1%}")

## 5. Leaky ReLU al rescate

Con pendiente `α` en `x<0`, ninguna neurona queda totalmente en 0 → sin muerte.

In [ ]:
inp = keras.Input(shape=(784,))
h = layers.Dense(256, kernel_initializer="he_normal")(inp)
h = layers.LeakyReLU(negative_slope=0.1)(h)
leaky_net = keras.Model(inp, h)

act = leaky_net(batch).numpy()
print("neuronas muertas con Leaky ReLU:",
      f"{float(np.mean(act.mean(axis=0) == 0.0)):.1%}")

## Ejercicios

1. **Plot**: graficá las 6 activaciones en `x ∈ [-3, 3]`.
2. **Comparación empírica**: entrená el MLP `[256,128,64]` con cada activación (mismo init He, mismo LR) y compará `val_accuracy` a 15 épocas.
3. **Dying ReLU**: con LR alto (0.1) entrená con ReLU y contá neuronas muertas por capa; repetí con Leaky y verificá que baja.
4. **GELU vs ReLU en profundidad**: MLP de 12 capas; GELU suele ganar.

## Conclusiones

- **ReLU** es rápida y default histórico, pero sufre **dying ReLU**; **Leaky/ELU** lo mitigan.
- **GELU** es el estándar en Transformers (BERT, GPT, ViT); **Swish/SiLU** en EfficientNet.
- GELU/Swish/Mish son suaves y no monótonas: mejor comportamiento del gradiente que ReLU en redes profundas.
- Su costo es mayor (sigmoid/erf internos) pero negligible en GPU/TPU.
- La activación de la **última** capa depende del problema (linear/sigmoid/softmax), no del criterio anterior.